# EDA — PhysioNet/CinC 2019 Sepsis Challenge dataset

Exploration only. Nothing here is imported by `src/` — see `src/data/ingest.py` for the function this notebook calls (CLAUDE.md: notebooks call src functions, never reimplement them).

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config
from src.data import ingest

df = ingest.load_all_patients()
df.shape

## 1. Class balance (patient level and row level)

In [ ]:
patient_label = df.groupby("patient_id")[config.LABEL_COLUMN].max()
n_patients = len(patient_label)
n_septic_patients = int(patient_label.sum())
row_prevalence = df[config.LABEL_COLUMN].mean()

print(f"Patients: {n_patients:,}")
print(f"Patients who ever develop sepsis: {n_septic_patients:,} ({100*n_septic_patients/n_patients:.2f}%)")
print(f"Row-level (patient-hour) positive rate: {100*row_prevalence:.2f}%")

## 2. Missingness per feature

In [ ]:
feature_cols = config.VITAL_COLUMNS + config.LAB_COLUMNS
missing_rate = df[feature_cols].isna().mean().sort_values(ascending=False)
missing_rate

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
missing_rate.plot(kind="barh", ax=ax)
ax.set_xlabel("Fraction missing")
ax.set_title("Missingness per feature, patient-hour level")
plt.tight_layout()
fig.savefig("../docs/eda_missingness.png", dpi=120)
plt.show()

## 3. Distribution of vitals

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, config.VITAL_COLUMNS):
    df[col].dropna().plot(kind="hist", bins=50, ax=ax, alpha=0.8)
    ax.set_title(col)
plt.tight_layout()
fig.savefig("../docs/eda_vitals_distribution.png", dpi=120)
plt.show()

## 4. Hours-to-onset distribution for positive cases

In [ ]:
septic_ids = patient_label[patient_label == 1].index
onset_hours = df[df["patient_id"].isin(septic_ids)].groupby("patient_id").apply(
    lambda g: g.loc[g[config.LABEL_COLUMN] == 1, "ICULOS"].min()
)
stay_length = df[df["patient_id"].isin(septic_ids)].groupby("patient_id")["ICULOS"].max()

fig, ax = plt.subplots(figsize=(8, 5))
onset_hours.plot(kind="hist", bins=40, ax=ax)
ax.set_xlabel("ICU hour of sepsis onset")
ax.set_title("Distribution of onset hour among septic patients")
plt.tight_layout()
fig.savefig("../docs/eda_onset_hour.png", dpi=120)
plt.show()

print(onset_hours.describe())

## 5. Stay-length distribution (all patients) — informs MAX_SEQUENCE_LENGTH

In [ ]:
all_stay_lengths = df.groupby("patient_id")["ICULOS"].max()
print(all_stay_lengths.describe())
print("99th percentile:", all_stay_lengths.quantile(0.99))
print("99.9th percentile:", all_stay_lengths.quantile(0.999))
print("max:", all_stay_lengths.max())